# Data

In [ ]:
# Cell 1
#!ls -la /root/.keras/datasets/plucad1.zip/
#!rm -r /root/.keras/datasets/besbos4
#!rm -rf /content/sample_data/epochs/*
import tensorflow.compat.v1 as tf

from tensorflow.keras import layers

print(tf.version.VERSION)
print(tf.keras.__version__)

# Data Preparation
- start this always before any training and validation, it loads all the data to the enviroment
- all the variables will be then accessible also in other cells

In [ ]:
import tensorflow as tf
import pathlib
import matplotlib.pyplot as plt
import os
import glob
import numpy as np
import PIL
import sys
import PIL.Image as Image
from PIL import ImageOps
import random
from scipy import ndimage
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPool2D, GlobalMaxPool2D, TimeDistributed, GRU, Dense, Dropout
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

"""
Here is the configuration part of the dataset structure
the definition of the ventilation and the perfusion part
the path are to the google drive, but can be changed to local
as per standard we have 8 perfusion and 6 ventilation projections"
"""
class lungs_configuration:
  def __init__(self, perfusion:int):
    self.EPOCHS = 200
    self.INITIAL_LEARNING_RATE = 0.001
    self.FINAL_LEARNING_RATE = 0.0001
    self.BATCH_SIZE = 14
    self.MAIN_PATH = "/content/drive/MyDrive/Colab/PATH_TO_DATASET/"
    self.AUTOTUNE = tf.data.AUTOTUNE
    
    #these shapes can be changed based on the dataset,
    #default 256x256 images from the camera
    
    if perfusion == 1:
        self.INPUT_SHAPE = (8,256,256,1)
    else:
        self.INPUT_SHAPE = (6,256,256,1)

    if perfusion == 1:
       self.LOG_PATH = "/content/drive/MyDrive/Colab/PATH_TO_DATASET/PERF_LOGS"
    else:
       self.LOG_PATH = "/content/drive/MyDrive/Colab/PATH_TO_DATASET/VENTI_LOGS"
    if perfusion == 1:
       self.CHECK_POINT_PATH = "/content/sample_data/epochs/save_at_{epoch}_plung.keras" #perfusion model
    else:
       self.CHECK_POINT_PATH = "/content/sample_data/epochs/save_at_{epoch}_vlung.keras" #ventilation model

    if perfusion == 1:
       self.DATASET = "NAME_OF_PERFUSION_DATASET" #dataset for perfusion
    else:
       self.DATASET = "NAME_OF_VENTILATION_DATASET" #dataset for ventilation

    if perfusion == 1:
      self.IMAGE_COUNT = 8 #define the image sequence for perfusion
    else:
      self.IMAGE_COUNT = 6 #define the image sequence for ventilation

    if perfusion == 1:
      self.ABNORMAL_PATH = 'ACUTE_PE_PERF' # abnormal perfusion
      self.NORMAL_PATH = 'CHRON_PE_PERF' #normal perfusion
    else:
      self.ABNORMAL_PATH = 'abnormal_venti' # abnonrmal ventilation
      self.NORMAL_PATH = 'norm_venti'

    if perfusion == 1:
      self.TRAIN_SET_SIZE = NUMBER #SIZE OF THE TRAINING SET FOR PERFUSION
      self.VALIDATION_SET_SIZE = NUMBER #SIZE OF THE VALIDATION SET FOR PERFUSION
    else:
      self.TRAIN_SET_SIZE = NUMBER #SIZE OF THE TRAINING SET FOR VENTILATION
      self.VALIDATION_SET_SIZE = NUMBER #SIZE OF THE VALIDATION SET FOR VENTILATION

# here is the perusion or ventilation dataset used, change the 0 to 1 for perfusion or ventilation
conf = lungs_configuration(0) # 1 perfusion 0 ventilation

archiveTrain = tf.keras.utils.get_file(origin=f"file://{conf.MAIN_PATH}{conf.DATASET}.zip", extract=True)
dataDirTrain = pathlib.Path(os.path.join(pathlib.Path(archiveTrain).with_suffix('.zip'), conf.DATASET))

abnormal_scan_paths = [
    os.path.join(conf.MAIN_PATH, str(dataDirTrain/conf.ABNORMAL_PATH/"train"/"*"), x)
    for x in glob.glob(str(dataDirTrain/conf.ABNORMAL_PATH/"train"/"*"))
]

normal_scan_paths = [
    os.path.join(conf.MAIN_PATH, str(dataDirTrain/conf.NORMAL_PATH/"train"/"*"), x)
    for x in glob.glob(str(dataDirTrain/conf.NORMAL_PATH/"train"/"*"))
]

abnormal_scan_paths_val = [
    os.path.join(conf.MAIN_PATH, str(dataDirTrain/conf.ABNORMAL_PATH/"validation"/"*"), x)
    for x in glob.glob(str(dataDirTrain/conf.ABNORMAL_PATH/"validation"/"*"))
]

normal_scan_paths_val = [
    os.path.join(conf.MAIN_PATH, str(dataDirTrain/conf.NORMAL_PATH/"validation"/"*"), x)
    for x in glob.glob(str(dataDirTrain/conf.NORMAL_PATH/"validation"/"*"))
]

def normalize(img):
    img = (img - np.min(img)) / (np.max(img) - np.min(img))
    return img.astype(np.float32)

def process_image(path):
  img = Image.open(path)
  img2 = ImageOps.grayscale(img)
  tmp1 = np.asarray(img2);
  tmp2 = normalize(tmp1)
  return tmp2
"""
This function creates the the arrays form the sequence of the images,
the images are exported so they create a sequence of 6 or 8 images
and are correctly fed into the array
"""
def create_array_sequence(scan_path,conf):
  data=[]
  for path in scan_path:
    lst = os.listdir(path)
    lst.sort()
    tmp = []
    image_index = 0
    image_count = len(lst)
    for index in range(conf.IMAGE_COUNT):
      if image_count == conf.IMAGE_COUNT:
        tmp.append(process_image(os.path.join(path,lst[index])))
      elif image_index < image_count:
        tmp.append(process_image(os.path.join(path,lst[image_index])))
      else:
        #if there are studies which do not have some projections these were excluded,
        #this is a fallback mechanism in case the export was broken
        tmp.append(np.zeros((256,256)))
    image_index+=1
    data.append(np.array(tmp))
  return np.array(data)

abnormal = create_array_sequence(abnormal_scan_paths,conf)
normal = create_array_sequence(normal_scan_paths,conf)
abnormal_val = create_array_sequence(abnormal_scan_paths_val,conf)
normal_val = create_array_sequence(normal_scan_paths_val,conf)

print("Abnormal train scans", abnormal.shape[0])
print("Normal train scans", normal.shape[0])

print("Abnormal validation scans", abnormal_val.shape[0])
print("Normal validation scans", normal_val.shape[0])

def train_preprocessing (volume, label):
  print("before",volume.shape)
  volume =tf.expand_dims(volume, axis=3)
  print("after",volume.shape)
  return volume, label

def val_preprocessing (volume, label):
  volume =tf.expand_dims(volume, axis=3)
  return volume, label

abnormal_labels = np.array([1 for _  in range(len(abnormal))])
normal_labels = np.array([0 for _  in range(len(normal))])
#validation labels
abnormal_labels_val = np.array([1 for _  in range(len(abnormal_val))])
normal_labels_val = np.array([0 for _  in range(len(normal_val))])

print("samples for training:", conf.TRAIN_SET_SIZE)
print("samples for validation:", conf.VALIDATION_SET_SIZE)
# this creates the classes and labels to it
x_train = np.concatenate((abnormal,normal),axis=0)
y_train = np.concatenate((abnormal_labels,normal_labels),axis=0)
# validation part
x_val = np.concatenate((abnormal_val,normal_val),axis=0)
y_val = np.concatenate((abnormal_labels_val,normal_labels_val),axis=0)

# the final shape of the the classes
print("shape xtrain:",x_train.shape)
print("shape ytrain:",y_train.shape)
print("shape xval:",x_val.shape)
print("shape yval:",y_val.shape)

#creating tensorflow dataset
train_loader = tf.data.Dataset.from_tensor_slices((x_train, y_train))
val_loader = tf.data.Dataset.from_tensor_slices((x_val, y_val))

train_dataset = (
    train_loader.shuffle(len(x_train),seed=236)
    #.map(augment_3d_volume) #in case of extra augmentation
    .map(train_preprocessing)
    .batch(conf.BATCH_SIZE)
    .prefetch(conf.AUTOTUNE)
)

def sample_count(train_dataset)->int:
  samples=0
  for x, y in train_dataset:
    print(x.shape, y.shape)
    samples += y.shape[0]
  print("Samples per train dataset:", samples)
  return samples

val_dataset = (
    val_loader.shuffle(len(x_val),seed=100)
    .map(val_preprocessing)
    .batch(conf.BATCH_SIZE)
    .prefetch(conf.AUTOTUNE)
)

data = train_dataset.take(2)
images,labels = list(data)[0]

images =images.numpy()
# print out images to see what is trained
for index in range(conf.IMAGE_COUNT):
  #print("image shape:",images[1][index].shape)
  image = images[0][index]
  plt.imshow(image,cmap="gray")
  plt.show()

# Training part
- start only after preparation section was run

In [ ]:
# this can be used for on the fly augmentation.
def augment_3d_volume(volume, label):
    """
    Applies data augmentation to a 5D/4D tensor safely without 2D layer crashes.
    Expects volume shape: (Depth, Height, Width, 1)
    """
    # 1. Random Left-Right Flip (Careful: Only use if laterality isn't a strict classification feature)
    #volume = tf.image.random_flip_left_right(volume)
    #volume = tf.image.rot90(volume)

    # 2. 3D Translation (Shift the lungs slightly up/down/left/right)
    # dx, dy, dz shifts
    dx = tf.random.uniform([], -2, 3, dtype=tf.int32)
    dy = tf.random.uniform([], -2, 3, dtype=tf.int32)

    # Roll tensor along axes to simulate shifting positioning in the SPECT camera
    volume = tf.roll(volume, shift=[dx, dy], axis=[1, 2])

    # 3. Add slight random contrast adjustments (simulates tracer dosage/scan timing variations)
    volume = tf.image.random_contrast(volume, lower=0.8, upper=1.2)

    return volume, label

pool_size = (1, 2, 2)
def se_attention_3d(inputs, ratio=4):
    """
    3D Squeeze-and-Excitation Channel Attention Block.
    Focuses on important feature channels dynamically.
    """
    filters = inputs.shape[-1]

    # Squeeze: Global Average Pool over the spatial dimensions
    se = tf.keras.layers.GlobalAveragePooling3D()(inputs)

    # Excite: Two dense layers to learn channel-wise weights
    # We use max(1, ...) to ensure the bottleneck never drops below 1 neuron
    se = tf.keras.layers.Dense(max(1, filters // ratio), activation='relu', use_bias=False)(se)
    se = tf.keras.layers.Dense(filters, activation='sigmoid', use_bias=False)(se)

    # Reshape weights so they can broadcast across the (Depth, Height, Width) dimensions
    se = tf.keras.layers.Reshape((1, 1, 1, filters))(se)

    # Scale the original inputs with the learned attention weights
    x = tf.keras.layers.Multiply()([inputs, se])
    return x

def get_model(shape=conf.INPUT_SHAPE, pool_size=(2, 2, 2)):
    """Build a 3D convolutional neural network model with Attention."""

    inputs = tf.keras.Input(shape=shape)
    
    reg = tf.keras.regularizers.l2(1e-4)

    # Block 1
    
    x = tf.keras.layers.Conv3D(filters=4, kernel_size=(3,3,3), activation="relu", padding="same", kernel_regularizer=reg)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=pool_size)(x)

    # Block 2
    x = tf.keras.layers.Conv3D(filters=8, kernel_size=(3,3,3), activation="relu", padding="same", kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=(1, 2, 2))(x)
    x = se_attention_3d(x, ratio=4)

    # Block 3 + Attention
    x = tf.keras.layers.Conv3D(filters=16, kernel_size=(3,3,3), activation="relu", padding="same", kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=(1, 2, 2))(x)


    # Block 4 + Attention
    x = tf.keras.layers.Conv3D(filters=32, kernel_size=(3,3,3), activation="relu", padding="same", kernel_regularizer=reg)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=(1, 2, 2))(x)
    x = se_attention_3d(x, ratio=4) # <-- Attention placed here

    # Classification Head
    x = tf.keras.layers.GlobalAveragePooling3D()(x)
    # you can also try different Dense layers and Dropout rates here to see if it improves performance
    #x = tf.keras.layers.Dense(units=512, activation="relu")(x)
    #x = tf.keras.layers.Dropout(0.5)(x)

    x = tf.keras.layers.Dense(units=64, activation="relu")(x)
    #x = tf.keras.layers.Dense(units=32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.6)(x)

    #x = tf.keras.layers.Dense(units=128, activation="relu")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    # Define the model.
    model = tf.keras.Model(inputs, outputs, name="3dcnn_with_attention")
    return model

#calculating the decay steps per epoch based on samples and batch size
def count_steps_per_epoch(conf, train_dataset, batch_size):
  learning_rate_decay_factor = (conf.FINAL_LEARNING_RATE / conf.INITIAL_LEARNING_RATE)**(1/conf.EPOCHS)
  steps_per_epoch = int(sample_count(train_dataset)/batch_size)
  print("steps per epoch:",steps_per_epoch)
  return learning_rate_decay_factor,steps_per_epoch

learning_rate_decay_factor,steps_per_epoch = count_steps_per_epoch(conf,train_dataset,conf.BATCH_SIZE)

#different learning rate schedulers can be used, the exponential decay is used here

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
                initial_learning_rate=conf.INITIAL_LEARNING_RATE,
                decay_steps=steps_per_epoch,
                decay_rate=learning_rate_decay_factor,
                staircase=True)

lr_scheduler2 = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',    # Watch the validation loss
    factor=0.5,            # Cut the learning rate in half (multiply by 0.5) when triggered
    patience=8,            # Wait 8 epochs of no improvement before reducing
    min_lr=1e-6,           # Never let the learning rate drop below this floor
    verbose=1              # Print a message to the console when the rate changes
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_3dcnn_model.keras',
    monitor='val_loss',
    save_best_only=True,   # Only overwrite the file if the model gets BETTER
    mode='min',            # We want the minimum validation loss
    verbose=1
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=25,           # Give up after 25 epochs of zero progress
    restore_best_weights=True # Automatically load the best weights into memory when done
)

model = get_model()

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    #optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    optimizer=tf.keras.optimizers.Adam(),
    metrics = [
          tf.keras.metrics.BinaryAccuracy(name="accuracy"),
          tf.keras.metrics.AUC(name="auc", multi_label=False),
          tf.keras.metrics.TruePositives(name='tp'),
          tf.keras.metrics.FalsePositives(name='fp'),
          tf.keras.metrics.TrueNegatives(name='tn'),
          tf.keras.metrics.FalseNegatives(name='fn'),
          tf.keras.metrics.Precision(name='precision'),
          tf.keras.metrics.Recall(name='recall'),
          tf.keras.metrics.SensitivityAtSpecificity(0.5, name='sensitivity'),
          tf.keras.metrics.SpecificityAtSensitivity(0.5, name='specificity'),
    ],
    #run_eagerly=False,
)

model.summary()

# Define callbacks.
callbacks = [
    tf.keras.callbacks.TensorBoard(log_dir=conf.LOG_PATH,write_graph=False),
    #early_stopping,
    checkpoint,
    lr_scheduler2
    # can be activated to stop if the training does not perform well
    #tf.keras.callbacks.EarlyStopping(patience=5, start_from_epoch=35,restore_best_weights=True)
]

# Train the model, doing validation at the end of each epoch
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=conf.EPOCHS,
    batch_size=conf.BATCH_SIZE,
    shuffle=True,
    verbose=1,
    callbacks=callbacks,
)

# Tensorboard section
- here the training observations can be done

In [ ]:
#%load_ext tensorboard
%reload_ext tensorboard
#!kill 19207
%tensorboard --logdir "/content/drive/MyDrive/Colab/pluca/logs1_lung"

# Validation and confidence interval section
- run this only after the preparation section was loaded to the session
- here are also Confident intervals calculated based on Clopper-Pearson

In [ ]:
from statsmodels.stats.proportion import proportion_confint
from sklearn.metrics import confusion_matrix

model =  tf.keras.models.load_model("/content/drive/MyDrive/PATH_TO_SAVED_MODEL/FILENAME.keras")
#model.summary()
# if model from Colab is used it can be saved to your drive
#model.save("/content/drive/MyDrive/Colab/PATH_TO_SAVED_MODEL/FILENAME.keras")

def calculate_predictions(model, x_val, y_val):
  res = []
  for index in range(x_val.shape[0]):
    prediction = model.predict(np.expand_dims(x_val[index], axis=0))
    print("Real state:",y_val[index])
    tmp = [y_val[index]]
    scores = [1 - prediction[0], prediction[0]]
    tmp.append(np.argmax(scores))
    class_names = ["normal", "abnormal"]
    for score, name in zip(scores, class_names):
        print(
            "This model is %.2f percent confident that Perfusion scan is %s"
            % ((100 * score), name)
        )
        tmp.append((100*score))
        tmp.append(name)
    res.append(tmp)

  rows = len(res)
  print("Unseen validation sample", rows)
  print(f"real,detected,normal_%,normal_label,abnormal_%,abnormal_label")
  for index in range(rows):
      print(f"{res[index][0]},{res[index][1]},{res[index][2]},{res[index][3]},{res[index][4]},{res[index][5]}")


y_true_list = []
y_pred_list = []

print("Running predictions on validation data...")
for x_batch, y_batch in val_dataset:
    # Model predicts a single probability (0.0 to 1.0) due to sigmoid activation
    print(x_batch.shape[0])
    preds = model.predict(x_batch, verbose=0)
    print(preds)
    y_true_list.extend(y_batch.numpy())
    y_pred_list.extend(preds)

# Convert to numpy arrays and flatten to ensure they are strictly 1D
# e.g., changing shape from (samples, 1) to (samples,)
print ("true list",y_true_list)
y_true = np.array(y_true_list).flatten()
y_pred = np.array(y_pred_list).flatten()

# 1. Convert your predicted probabilities to hard classes (0 or 1) using a 0.5 threshold
y_pred_classes = (y_pred >= 0.5).astype(int)

# 2. Get True Negatives, False Positives, False Negatives, True Positives
tn, fp, fn, tp = confusion_matrix(y_true, y_pred_classes, labels=[0, 1]).ravel()
# Total actual positives (Abnormal) and actual negatives (Normal)
total_positives = tp + fn 
total_negatives = tn + fp 

# 3. Calculate metrics
sensitivity = tp / total_positives if total_positives > 0 else 0
specificity = tn / total_negatives if total_negatives > 0 else 0
accuracy = (tp + tn) / (total_positives + total_negatives)

# 4. Calculate Exact Binomial Confidence Intervals (Clopper-Pearson method via 'beta')
sens_lower, sens_upper = proportion_confint(count=tp, nobs=total_positives, alpha=0.05, method='beta')
spec_lower, spec_upper = proportion_confint(count=tn, nobs=total_negatives, alpha=0.05, method='beta')
acc_lower, acc_upper = proportion_confint(count=(tp+tn), nobs=(total_positives+total_negatives), alpha=0.05, method='beta')

# 5. Print the Clinical Report
print("-" * 50)
print(f"Model Evaluation (N={len(y_true)} Scans):")
print("-" * 50)
print(f"Confusion Matrix: TP={tp}, FN={fn}, TN={tn}, FP={fp}")
print(f"Accuracy:    {accuracy*100:5.1f}%  | 95% CI: [{acc_lower*100:5.1f}%, {acc_upper*100:5.1f}%]")
print(f"Sensitivity: {sensitivity*100:5.1f}%  | 95% CI: [{sens_lower*100:5.1f}%, {sens_upper*100:5.1f}%]")
print(f"Specificity: {specificity*100:5.1f}%  | 95% CI: [{spec_lower*100:5.1f}%, {spec_upper*100:5.1f}%]")
print("-" * 50)

